In [1]:
import sys
sys.path.insert(0, '..')

import requests
import pandas as pd
import numpy as np
import io
import time
import warnings
import pyarrow as pa
import pyarrow.parquet as pq
from pathlib import Path
from sodapy import Socrata

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

# Rutas
RAW          = Path('../data/01_raw')
INTERMEDIATE = Path('../data/02_intermediate')
PRIMARY      = Path('../data/03_primary')
INTERMEDIATE.mkdir(parents=True, exist_ok=True)
PRIMARY.mkdir(parents=True, exist_ok=True)

print('✅ Librerías cargadas')
print(f'✅ RAW:          {RAW}')
print(f'✅ INTERMEDIATE: {INTERMEDIATE}')
print(f'✅ PRIMARY:      {PRIMARY}')

✅ Librerías cargadas
✅ RAW:          ..\data\01_raw
✅ INTERMEDIATE: ..\data\02_intermediate
✅ PRIMARY:      ..\data\03_primary


In [2]:
def inspeccionar(df, nombre):
    """Reporte completo de calidad de datos para un DataFrame."""
    print(f'\n{"="*60}')
    print(f'📋 {nombre}')
    print(f'{"="*60}')
    print(f'Shape: {df.shape[0]:,} filas × {df.shape[1]} columnas')

    # Tipos
    print(f'\nTipos de datos:')
    for col, dtype in df.dtypes.items():
        print(f'  {col}: {dtype}')

    # Nulos
    nulos = df.isnull().sum()
    nulos_pct = (nulos / len(df) * 100).round(2)
    df_nulos = pd.DataFrame({'nulos': nulos, 'pct_%': nulos_pct})
    df_nulos = df_nulos[df_nulos['nulos'] > 0].sort_values('pct_%', ascending=False)
    if len(df_nulos) > 0:
        print(f'\n🔍 Nulos por columna:')
        print(df_nulos.to_string())
    else:
        print(f'\n✅ Sin nulos')

    # Duplicados
    dup_exactos = df.duplicated().sum()
    print(f'\n🔍 Duplicados exactos: {dup_exactos:,}')

    return df_nulos

def analizar_outliers(df, columnas_num, nombre):
    """Detecta outliers con método IQR."""
    print(f'\n🔍 Outliers ({nombre}):')
    for col in columnas_num:
        if col not in df.columns:
            continue
        serie = pd.to_numeric(df[col], errors='coerce').dropna()
        if len(serie) == 0:
            continue
        Q1 = serie.quantile(0.25)
        Q3 = serie.quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR
        outliers = serie[(serie < lower) | (serie > upper)]
        print(f'  {col}: {len(outliers):,} outliers '
              f'| rango válido [{lower:.2f}, {upper:.2f}] '
              f'| min: {serie.min():.2f} | max: {serie.max():.2f}')

def validar_divipola(df, col_div, nombre):
    """Valida formato DIVIPOLA en una columna."""
    div = df[col_div].astype(str)
    invalidos = div[div.str.len() != 5]
    no_numericos = div[~div.str.isdigit()]
    print(f'\n🔍 Validación DIVIPOLA ({nombre}):')
    print(f'  Total: {len(div):,}')
    print(f'  Longitud != 5: {len(invalidos):,}')
    print(f'  No numéricos: {len(no_numericos):,}')
    if len(invalidos) > 0:
        print(f'  Muestra inválidos: {invalidos.head(5).tolist()}')

def guardar_intermedio(df, nombre):
    """Guarda DataFrame en formato Parquet en carpeta intermedia."""
    ruta = INTERMEDIATE / f'{nombre}.parquet'
    pq.write_table(pa.Table.from_pandas(df), ruta, compression='snappy')
    mb = ruta.stat().st_size / (1024*1024)
    print(f'\n💾 Guardado: {nombre}.parquet ({mb:.1f} MB) — {len(df):,} filas')
    return ruta

def descargar_soda(dataset_id, nombre, columnas=None, page_size=50000):
    """Descarga un dataset completo de la SODA API con paginación."""
    client = Socrata("www.datos.gov.co", None, timeout=30)
    total = int(client.get(dataset_id, select="count(*)")[0]["count"])
    print(f'\n⬇️  Descargando {nombre} ({dataset_id}) — {total:,} registros')
    registros = []
    offset, pagina = 0, 1
    while offset < total:
        params = {"limit": page_size, "offset": offset, "order": ":id"}
        if columnas:
            params["select"] = ", ".join(columnas)
        registros.extend(client.get(dataset_id, **params))
        print(f'   Página {pagina} — {min(offset+page_size,total):,}/{total:,}')
        offset += page_size
        pagina += 1
        time.sleep(0.2)
    client.close()
    df = pd.DataFrame.from_records(registros)
    print(f'   ✅ {len(df):,} registros descargados')
    return df

print('✅ Funciones de utilidad listas')

✅ Funciones de utilidad listas


#  EVA

In [3]:
print('='*60)
print('FUENTE 1 — EVA 2019-2024')
print('='*60)

# Descarga
df_eva_raw = descargar_soda(
    dataset_id="uejq-wxrr",
    nombre="EVA",
    columnas=[
        "c_digo_dane_departamento","departamento",
        "c_digo_dane_municipio","municipio",
        "grupo_cultivo","cultivo","a_o",
        "rea_sembrada","rea_cosechada",
        "producci_n","rendimiento","ciclo_del_cultivo",
    ]
)

# Inspección
inspeccionar(df_eva_raw, "EVA RAW")

# DIVIPOLA
validar_divipola(df_eva_raw, "c_digo_dane_municipio", "EVA")

# Duplicados por llave
dup_llave = df_eva_raw.duplicated(subset=["c_digo_dane_municipio","cultivo","a_o"]).sum()
print(f'\n🔍 Duplicados por municipio+cultivo+año: {dup_llave:,}')

# Valores únicos clave
print(f'\n🔍 Años disponibles: {sorted(df_eva_raw["a_o"].unique().tolist())}')
print(f'🔍 Cultivos únicos: {df_eva_raw["cultivo"].nunique():,}')
print(f'🔍 Grupos únicos: {df_eva_raw["grupo_cultivo"].nunique():,}')

# Outliers
df_eva_raw_num = df_eva_raw.copy()
for col in ["rea_sembrada","rea_cosechada","producci_n","rendimiento"]:
    df_eva_raw_num[col] = pd.to_numeric(df_eva_raw_num[col], errors='coerce')
analizar_outliers(df_eva_raw_num, ["rea_sembrada","rea_cosechada","producci_n","rendimiento"], "EVA")

# Limpieza
print(f'\n🧹 Limpiando EVA...')
df_eva = df_eva_raw.copy()
df_eva = df_eva.rename(columns={
    "c_digo_dane_municipio":    "cod_municipio_raw",
    "c_digo_dane_departamento": "cod_depto_raw",
    "a_o":                      "anio",
    "rea_sembrada":             "area_sembrada_ha",
    "rea_cosechada":            "area_cosechada_ha",
    "producci_n":               "produccion_ton",
    "rendimiento":              "rendimiento_ton_ha",
})
df_eva["divipola"] = df_eva["cod_municipio_raw"].astype(str).str.zfill(5)
df_eva["anio"] = pd.to_numeric(df_eva["anio"], errors="coerce").astype("Int64")
for col in ["area_sembrada_ha","area_cosechada_ha","produccion_ton","rendimiento_ton_ha"]:
    df_eva[col] = pd.to_numeric(df_eva[col], errors="coerce")

antes = len(df_eva)
df_eva = df_eva.dropna(subset=["divipola","anio"])
df_eva = df_eva[df_eva["divipola"].str.len() == 5]
df_eva = df_eva.drop_duplicates(subset=["divipola","cultivo","anio"])
print(f'  Filas eliminadas: {antes - len(df_eva):,}')

# Validación final
print(f'\n✅ EVA limpia:')
print(f'  Registros: {len(df_eva):,}')
print(f'  Municipios: {df_eva["divipola"].nunique():,}')
print(f'  Años: {sorted(df_eva["anio"].dropna().unique().tolist())}')
print(f'  Cultivos: {df_eva["cultivo"].nunique():,}')
print(df_eva[["divipola","municipio","cultivo","anio","produccion_ton","rendimiento_ton_ha"]].head(3).to_string())

guardar_intermedio(df_eva, "eva_limpia")

FUENTE 1 — EVA 2019-2024

⬇️  Descargando EVA (uejq-wxrr) — 141,073 registros
   Página 1 — 50,000/141,073
   Página 2 — 100,000/141,073
   Página 3 — 141,073/141,073
   ✅ 141,073 registros descargados

📋 EVA RAW
Shape: 141,073 filas × 12 columnas

Tipos de datos:
  c_digo_dane_departamento: object
  departamento: object
  c_digo_dane_municipio: object
  municipio: object
  grupo_cultivo: object
  cultivo: object
  a_o: object
  rea_sembrada: object
  rea_cosechada: object
  producci_n: object
  rendimiento: object
  ciclo_del_cultivo: object

✅ Sin nulos

🔍 Duplicados exactos: 4,612

🔍 Validación DIVIPOLA (EVA):
  Total: 141,073
  Longitud != 5: 0
  No numéricos: 0

🔍 Duplicados por municipio+cultivo+año: 47,587

🔍 Años disponibles: ['2019', '2020', '2021', '2022', '2023', '2024']
🔍 Cultivos únicos: 166
🔍 Grupos únicos: 8

🔍 Outliers (EVA):
  rea_sembrada: 21,302 outliers | rango válido [-125.50, 224.58] | min: 0.00 | max: 60000.00
  rea_cosechada: 21,731 outliers | rango válido [-110

WindowsPath('../data/02_intermediate/eva_limpia.parquet')

Duplicados por municipio+cultivo+año: 47.587 — esto es alto. Significa que hay múltiples reportes del mismo cultivo en el mismo municipio y año, probablemente por diferentes períodos de cosecha o variedades. La deduplicación los eliminó correctamente quedando con el primer registro por llave.

Outliers en producción — el rango IQR marca muchos outliers porque hay cultivos con producción masiva (café, caña, arroz) vs cultivos de subsistencia. Estos NO se eliminan — son datos reales, solo se documentan.

# Frontera Agrícola

In [4]:
print('='*60)
print('FUENTE 2 — Frontera Agrícola')
print('='*60)

df_frontera_raw = descargar_soda(
    dataset_id="fyc7-sbtz",
    nombre="Frontera Agrícola",
    columnas=["municipio","departamen","cod_depart","cod_dane_m","tipo_front","area_ha"]
)

inspeccionar(df_frontera_raw, "Frontera Agrícola RAW")
validar_divipola(df_frontera_raw, "cod_dane_m", "Frontera")

print(f'\n🔍 Tipos de frontera: {df_frontera_raw["tipo_front"].value_counts().to_dict()}')
df_frontera_raw["area_ha_num"] = pd.to_numeric(df_frontera_raw["area_ha"], errors="coerce")
analizar_outliers(df_frontera_raw, ["area_ha_num"], "Frontera")

print(f'\n🧹 Limpiando Frontera Agrícola...')
df_frontera = df_frontera_raw.copy()
df_frontera["divipola"] = df_frontera["cod_dane_m"].astype(str).str.zfill(5)
df_frontera["area_ha"] = pd.to_numeric(df_frontera["area_ha"], errors="coerce")
df_frontera = df_frontera.rename(columns={"departamen":"departamento","tipo_front":"tipo_frontera"})
antes = len(df_frontera)
df_frontera = df_frontera.dropna(subset=["divipola","area_ha"])
df_frontera = df_frontera[df_frontera["divipola"].str.len() == 5]
print(f'  Filas eliminadas: {antes - len(df_frontera):,}')

print(f'\n✅ Frontera limpia:')
print(f'  Registros: {len(df_frontera):,}')
print(f'  Municipios: {df_frontera["divipola"].nunique():,}')
print(f'  Tipos: {df_frontera["tipo_frontera"].value_counts().to_dict()}')

guardar_intermedio(df_frontera, "frontera_agricola_limpia")

FUENTE 2 — Frontera Agrícola

⬇️  Descargando Frontera Agrícola (fyc7-sbtz) — 455,143 registros
   Página 1 — 50,000/455,143
   Página 2 — 100,000/455,143
   Página 3 — 150,000/455,143
   Página 4 — 200,000/455,143
   Página 5 — 250,000/455,143
   Página 6 — 300,000/455,143
   Página 7 — 350,000/455,143
   Página 8 — 400,000/455,143
   Página 9 — 450,000/455,143
   Página 10 — 455,143/455,143
   ✅ 455,143 registros descargados

📋 Frontera Agrícola RAW
Shape: 455,143 filas × 6 columnas

Tipos de datos:
  municipio: object
  departamen: object
  cod_depart: object
  cod_dane_m: object
  tipo_front: object
  area_ha: object

✅ Sin nulos

🔍 Duplicados exactos: 68,868

🔍 Validación DIVIPOLA (Frontera):
  Total: 455,143
  Longitud != 5: 0
  No numéricos: 0

🔍 Tipos de frontera: {'Condicionada': 243817, 'No condicionada': 211326}

🔍 Outliers (Frontera):
  area_ha_num: 82,171 outliers | rango válido [-0.84, 1.46] | min: 0.00 | max: 873571.08

🧹 Limpiando Frontera Agrícola...
  Filas eliminadas

WindowsPath('../data/02_intermediate/frontera_agricola_limpia.parquet')

Cobertura perfecta: 1.122 municipios — cubre todos incluyendo ANM. ✅

68.868 duplicados exactos — son polígonos geoespaciales repetidos. No se eliminaron porque en datasets geoespaciales un municipio tiene múltiples polígonos (uno por cada zona de frontera). Es correcto mantenerlos — en la Fase de feature engineering los agregaremos sumando area_ha por municipio y tipo.

Outliers en área — rango IQR muy estrecho porque la mayoría son polígonos pequeños, pero hay algunos enormes (873.571 ha). Son datos reales de municipios grandes como Vichada o Guainía.

Solo dos tipos: Condicionada y No condicionada — más simple de lo esperado. Para el IRA el área "No condicionada" es la que directamente puede producir alimentos.

# Distritos de Riego

In [5]:
print('='*60)
print('FUENTE 3 — Distritos de Riego')
print('='*60)

df_dist_raw = descargar_soda(
    dataset_id="rtxu-twjm",
    nombre="Distritos de Riego",
    columnas=[
        "cod_dane_depto","departamento_ubicacion","cod_dane_municipio",
        "municipio_ubicacion_distrito","nombre_del_distrito",
        "escala_distrito","tipo_de_distrito","distrito_en_operacion_s_n",
        "area_bruta_ha","area_neta_inicial_beneficiada","numero_familias",
    ]
)

inspeccionar(df_dist_raw, "Distritos RAW")
validar_divipola(df_dist_raw, "cod_dane_municipio", "Distritos")

print(f'\n🔍 En operación: {df_dist_raw["distrito_en_operacion_s_n"].value_counts().to_dict()}')
print(f'🔍 Tipos: {df_dist_raw["tipo_de_distrito"].value_counts().to_dict()}')
df_dist_raw["area_bruta_ha_num"] = pd.to_numeric(df_dist_raw["area_bruta_ha"], errors="coerce")
analizar_outliers(df_dist_raw, ["area_bruta_ha_num"], "Distritos")

print(f'\n🧹 Limpiando Distritos...')
df_dist = df_dist_raw.copy()
df_dist["divipola"] = df_dist["cod_dane_municipio"].astype(str).str.strip().str.zfill(5)
df_dist["area_bruta_ha"] = pd.to_numeric(df_dist["area_bruta_ha"], errors="coerce")
df_dist["area_neta_inicial_beneficiada"] = pd.to_numeric(df_dist["area_neta_inicial_beneficiada"], errors="coerce")
df_dist["numero_familias"] = pd.to_numeric(df_dist["numero_familias"], errors="coerce")
df_dist = df_dist[df_dist["distrito_en_operacion_s_n"].str.upper() == "SI"].copy()
df_dist = df_dist.rename(columns={
    "municipio_ubicacion_distrito": "municipio",
    "departamento_ubicacion":       "departamento",
})
antes = len(df_dist)
df_dist = df_dist.dropna(subset=["divipola"])
df_dist = df_dist[df_dist["divipola"].str.len() == 5]
print(f'  Filas eliminadas: {antes - len(df_dist):,}')

print(f'\n✅ Distritos activos limpios:')
print(f'  Registros: {len(df_dist):,}')
print(f'  Municipios con riego: {df_dist["divipola"].nunique():,}')
print(f'  Tipos: {df_dist["tipo_de_distrito"].value_counts().to_dict()}')

guardar_intermedio(df_dist, "distritos_riego_limpia")

FUENTE 3 — Distritos de Riego

⬇️  Descargando Distritos de Riego (rtxu-twjm) — 833 registros
   Página 1 — 833/833
   ✅ 833 registros descargados

📋 Distritos RAW
Shape: 833 filas × 11 columnas

Tipos de datos:
  cod_dane_depto: object
  departamento_ubicacion: object
  cod_dane_municipio: object
  municipio_ubicacion_distrito: object
  nombre_del_distrito: object
  escala_distrito: object
  tipo_de_distrito: object
  distrito_en_operacion_s_n: object
  area_bruta_ha: object
  area_neta_inicial_beneficiada: object
  numero_familias: object

🔍 Nulos por columna:
                               nulos  pct_%
area_neta_inicial_beneficiada      2   0.24
numero_familias                    2   0.24
area_bruta_ha                      1   0.12

🔍 Duplicados exactos: 0

🔍 Validación DIVIPOLA (Distritos):
  Total: 833
  Longitud != 5: 75
  No numéricos: 0
  Muestra inválidos: ['5091', '5125', '5138', '5138', '5190']

🔍 En operación: {'SI': 580, 'NO': 253}
🔍 Tipos: {'Riego': 793, 'Riego y Drenaje'

WindowsPath('../data/02_intermediate/distritos_riego_limpia.parquet')

75 DIVIPOLA de 4 dígitos — la muestra confirma 5091, 5125, etc. El zfill(5) los corrige automáticamente a 05091, 05125. ✅ Ya está manejado en la limpieza.

580 activos de 833 — solo 278 municipios tienen infraestructura de riego activa. Los 826 restantes tendrán valor 0 en la tabla maestra, lo cual es información válida — ausencia de riego es un indicador de vulnerabilidad.

Sin duplicados — dataset limpio de origen. ✅

#  Informalidad y Finagro

In [6]:
print('='*60)
print('FUENTE 4 — Informalidad de Tierras')
print('='*60)

df_inf_raw = descargar_soda(
    dataset_id="hc6u-q778",
    nombre="Informalidad",
    columnas=["cod_depart","departamen","area_ha","informalid"]
)

inspeccionar(df_inf_raw, "Informalidad RAW")
print(f'\n🔍 Nota: dataset departamental (32 registros) — variable contextual en XGBoost')
df_inf_raw["informalid_num"] = pd.to_numeric(df_inf_raw["informalid"], errors="coerce")
analizar_outliers(df_inf_raw, ["informalid_num"], "Informalidad")

df_inf = df_inf_raw.copy()
df_inf["cod_depto"] = df_inf["cod_depart"].astype(str).str.zfill(2)
df_inf["indice_informalidad"] = pd.to_numeric(df_inf["informalid"], errors="coerce")
df_inf = df_inf.rename(columns={"departamen":"departamento"})
df_inf = df_inf.dropna(subset=["cod_depto","indice_informalidad"])

print(f'\n✅ Informalidad limpia:')
print(f'  Departamentos: {len(df_inf):,}')
print(f'  Rango: [{df_inf["indice_informalidad"].min():.3f}, {df_inf["indice_informalidad"].max():.3f}]')
print(df_inf[["cod_depto","departamento","indice_informalidad"]].head(5).to_string())

guardar_intermedio(df_inf, "informalidad_limpia")

print('\n' + '='*60)
print('FUENTE 5 — Créditos Finagro')
print('='*60)

df_fin_raw = descargar_soda(
    dataset_id="w3uf-w9ey",
    nombre="Finagro",
    columnas=["a_o","mes","id_munic","municipio_inversion","id_depto",
              "departamento_inversion","tipo_productor","colocacion","linea_de_produccion"]
)

inspeccionar(df_fin_raw, "Finagro RAW")
validar_divipola(df_fin_raw, "id_munic", "Finagro")

df_fin_raw["colocacion_num"] = pd.to_numeric(df_fin_raw["colocacion"], errors="coerce")
analizar_outliers(df_fin_raw, ["colocacion_num"], "Finagro")
dup_fin = df_fin_raw.duplicated(subset=["id_munic","a_o","mes","linea_de_produccion"]).sum()
print(f'\n🔍 Duplicados por municipio+año+mes+línea: {dup_fin:,}')
print(f'🔍 Tipos de productor: {df_fin_raw["tipo_productor"].value_counts().to_dict()}')
print(f'🔍 Años disponibles: {sorted(df_fin_raw["a_o"].unique().tolist())}')

print(f'\n🧹 Limpiando Finagro...')
df_fin = df_fin_raw.copy()
df_fin["divipola"] = df_fin["id_munic"].astype(str).str.zfill(5)
df_fin["anio"] = pd.to_numeric(df_fin["a_o"], errors="coerce").astype("Int64")
df_fin["mes"] = pd.to_numeric(df_fin["mes"], errors="coerce").astype("Int64")
df_fin["colocacion"] = pd.to_numeric(df_fin["colocacion"], errors="coerce")
df_fin = df_fin.rename(columns={
    "municipio_inversion": "municipio",
    "departamento_inversion": "departamento"
})
antes = len(df_fin)
df_fin = df_fin.dropna(subset=["divipola","colocacion"])
df_fin = df_fin[df_fin["divipola"].str.len() == 5]
df_fin = df_fin[df_fin["colocacion"] > 0]
print(f'  Filas eliminadas: {antes - len(df_fin):,}')

print(f'\n✅ Finagro limpio:')
print(f'  Registros: {len(df_fin):,}')
print(f'  Municipios: {df_fin["divipola"].nunique():,}')
print(f'  Colocación total: ${df_fin["colocacion"].sum():,.0f}')

guardar_intermedio(df_fin, "finagro_limpia")

FUENTE 4 — Informalidad de Tierras

⬇️  Descargando Informalidad (hc6u-q778) — 32 registros
   Página 1 — 32/32


   ✅ 32 registros descargados

📋 Informalidad RAW
Shape: 32 filas × 4 columnas

Tipos de datos:
  cod_depart: object
  departamen: object
  area_ha: object
  informalid: object

✅ Sin nulos

🔍 Duplicados exactos: 0

🔍 Nota: dataset departamental (32 registros) — variable contextual en XGBoost

🔍 Outliers (Informalidad):
  informalid_num: 1 outliers | rango válido [0.20, 0.82] | min: 0.16 | max: 0.68

✅ Informalidad limpia:
  Departamentos: 32
  Rango: [0.163, 0.679]
  cod_depto departamento  indice_informalidad
0        05    Antioquia                 0.51
1        08    Atlántico                 0.51
2        13      Bolívar                 0.52
3        15       Boyacá                 0.63
4        17       Caldas                 0.45

💾 Guardado: informalidad_limpia.parquet (0.0 MB) — 32 filas

FUENTE 5 — Créditos Finagro

⬇️  Descargando Finagro (w3uf-w9ey) — 1,907,696 registros
   Página 1 — 50,000/1,907,696
   Página 2 — 100,000/1,907,696
   Página 3 — 150,000/1,907,696
   Página

WindowsPath('../data/02_intermediate/finagro_limpia.parquet')

1. DIVIPOLA de 4 dígitos: 208.333 registros — igual que Distritos. El zfill(5) los corrige pero hay que verificar que quedó bien aplicado.

2. Duplicados críticos: 1.633.755 por municipio+año+mes+línea — esto es el 85% del dataset. El problema es que la llave de deduplicación es demasiado específica. Para el IRA no necesitamos el detalle por transacción, necesitamos el crédito total por municipio-año. La deduplicación se hará en feature engineering, no aquí.

3. Outlier extremo: $300.000.000.000 — hay un crédito de 300 billones de pesos que es claramente un error de datos. Hay que documentarlo y filtrarlo.

In [7]:
# Corrección Finagro — outlier extremo
print('🔍 Análisis outlier extremo Finagro:')
df_fin_outlier = df_fin[df_fin['colocacion'] > 1e11]
print(f'  Registros con colocación > $100.000.000.000: {len(df_fin_outlier):,}')
print(df_fin_outlier[['divipola','municipio','anio','colocacion','linea_de_produccion']].to_string())

# Filtrar outlier extremo — umbral: 10 billones (máximo razonable por transacción)
umbral = 10_000_000_000
antes = len(df_fin)
df_fin = df_fin[df_fin['colocacion'] <= umbral]
print(f'\n  Filas eliminadas por outlier extremo: {antes - len(df_fin):,}')
print(f'  Colocación máxima después del filtro: ${df_fin["colocacion"].max():,.0f}')
print(f'  Registros finales: {len(df_fin):,}')

# Verificar DIVIPOLA después del zfill
print(f'\n🔍 DIVIPOLA después de limpieza:')
div_invalidos = df_fin[df_fin['divipola'].str.len() != 5]
print(f'  DIVIPOLA longitud != 5: {len(div_invalidos):,}')
print(f'  Municipios únicos: {df_fin["divipola"].nunique():,}')

# Re-guardar
guardar_intermedio(df_fin, "finagro_limpia")

🔍 Análisis outlier extremo Finagro:
  Registros con colocación > $100.000.000.000: 42
        divipola         municipio  anio      colocacion              linea_de_produccion
18569      05001          MEDELLÍN  2021 297251733313.00          SERVICIOS DE APOYO (CT)
18787      05001          MEDELLÍN  2021 146113999997.00          SERVICIOS DE APOYO (CT)
79280      05001          MEDELLÍN  2021 129990401388.00          SERVICIOS DE APOYO (CT)
80979      08001      BARRANQUILLA  2021 158533333332.00          SERVICIOS DE APOYO (CT)
81017      11001      BOGOTÁ, D.C.  2021 100378560000.00             COMERCIALIZACION (I)
119293     76520           PALMIRA  2021 152062834924.00            COMERCIALIZACION (CT)
121062     08001      BARRANQUILLA  2021 115000000000.00          SERVICIOS DE APOYO (CT)
121254     08001      BARRANQUILLA  2021 158533333332.00          SERVICIOS DE APOYO (CT)
240686     76001              CALI  2021 184718796915.00            COMERCIALIZACION (CT)
263059     080

WindowsPath('../data/02_intermediate/finagro_limpia.parquet')

Hallazgo importante: Los outliers extremos son todos de líneas COMERCIALIZACION, SERVICIOS DE APOYO y TRANSFORMACIÓN en ciudades grandes (Bogotá, Medellín, Cali, Barranquilla). No son créditos a agricultores sino a empresas agroindustriales y comercializadoras. Para el IRA que mide capacidad productiva rural, su exclusión es metodológicamente correcta y defendible.

DIVIPOLA corregido: 0 inválidos — el zfill(5) funcionó perfectamente. ✅

1.112 municipios — cobertura alta para un dataset de crédito agropecuario.

# Estaciones y Normales IDEAM

In [8]:
print('='*60)
print('FUENTE 6 — Estaciones IDEAM')
print('='*60)

df_est_raw = descargar_soda(
    dataset_id="hp9r-jxuu",
    nombre="Estaciones IDEAM",
    columnas=["codigo","nombre","categoria","tecnologia","estado",
              "departamento","municipio","altitud","longitud","latitud"]
)

inspeccionar(df_est_raw, "Estaciones IDEAM RAW")
print(f'\n🔍 Estado: {df_est_raw["estado"].value_counts().to_dict()}')
print(f'🔍 Categoría: {df_est_raw["categoria"].value_counts().to_dict()}')

df_est_raw["latitud_num"] = pd.to_numeric(df_est_raw["latitud"], errors="coerce")
df_est_raw["longitud_num"] = pd.to_numeric(df_est_raw["longitud"], errors="coerce")

# Coordenadas fuera de Colombia
fuera_col = df_est_raw[
    ~(df_est_raw["latitud_num"].between(-4.5, 13.0) &
      df_est_raw["longitud_num"].between(-82.0, -66.0))
]
print(f'\n🔍 Estaciones fuera de Colombia: {len(fuera_col):,}')

print(f'\n🧹 Limpiando Estaciones IDEAM...')
df_est = df_est_raw.copy()
df_est["longitud"] = pd.to_numeric(df_est["longitud"], errors="coerce")
df_est["latitud"] = pd.to_numeric(df_est["latitud"], errors="coerce")
df_est["altitud"] = pd.to_numeric(df_est["altitud"], errors="coerce")
antes = len(df_est)
df_est = df_est.dropna(subset=["latitud","longitud"])
df_est = df_est[
    (df_est["latitud"].between(-4.5, 13.0)) &
    (df_est["longitud"].between(-82.0, -66.0))
]
print(f'  Filas eliminadas: {antes - len(df_est):,}')

print(f'\n✅ Estaciones limpias:')
print(f'  Total: {len(df_est):,}')
print(f'  Activas: {len(df_est[df_est["estado"]=="Activa"]):,}')
print(f'  Nota: DIVIPOLA se asignará por cruce geoespacial en pipeline_integration')

guardar_intermedio(df_est, "estaciones_ideam_limpia")

print('\n' + '='*60)
print('FUENTE 7 — Normales Climatológicas')
print('='*60)

df_norm_raw = descargar_soda(
    dataset_id="nsz2-kzcq",
    nombre="Normales Climatológicas",
    columnas=["periodo","par_metro","c_digo","categoria","estaci_n","municipio",
              "departamento","altitud_m","longitud","latitud",
              "ene","feb","mar","abr","may","jun",
              "jul","ago","sep","oct","nov","dic","anual"]
)

inspeccionar(df_norm_raw, "Normales RAW")
print(f'\n🔍 Parámetros: {df_norm_raw["par_metro"].value_counts().to_dict()}')
print(f'🔍 Períodos: {df_norm_raw["periodo"].unique().tolist()}')

MESES = ["ene","feb","mar","abr","may","jun","jul","ago","sep","oct","nov","dic"]
for col in MESES + ["anual"]:
    df_norm_raw[col] = pd.to_numeric(df_norm_raw[col], errors="coerce")

analizar_outliers(df_norm_raw, ["anual"], "Normales")

print(f'\n🧹 Limpiando Normales...')
df_norm = df_norm_raw.copy()
df_norm = df_norm.rename(columns={
    "par_metro":  "parametro",
    "c_digo":     "codigo_estacion",
    "estaci_n":   "estacion",
    "altitud_m":  "altitud",
})
df_norm["longitud"] = pd.to_numeric(df_norm["longitud"], errors="coerce")
df_norm["latitud"] = pd.to_numeric(df_norm["latitud"], errors="coerce")
antes = len(df_norm)
df_norm = df_norm.dropna(subset=["latitud","longitud"])
df_norm = df_norm[
    (df_norm["latitud"].between(-4.5, 13.0)) &
    (df_norm["longitud"].between(-82.0, -66.0))
]
print(f'  Filas eliminadas: {antes - len(df_norm):,}')

print(f'\n✅ Normales limpias:')
print(f'  Registros: {len(df_norm):,}')
print(f'  Estaciones únicas: {df_norm["codigo_estacion"].nunique():,}')
print(f'  Parámetros: {df_norm["parametro"].unique().tolist()}')

guardar_intermedio(df_norm, "normales_climatologicas_limpia")

FUENTE 6 — Estaciones IDEAM

⬇️  Descargando Estaciones IDEAM (hp9r-jxuu) — 9,685 registros
   Página 1 — 9,685/9,685


   ✅ 9,685 registros descargados

📋 Estaciones IDEAM RAW
Shape: 9,685 filas × 10 columnas

Tipos de datos:
  codigo: object
  nombre: object
  categoria: object
  tecnologia: object
  estado: object
  departamento: object
  municipio: object
  altitud: object
  longitud: object
  latitud: object

✅ Sin nulos

🔍 Duplicados exactos: 0

🔍 Estado: {'Activa': 5647, 'Suspendida': 3689, 'En Mantenimiento': 349}
🔍 Categoría: {'Pluviométrica': 3822, 'Limnimétrica': 1962, 'Limnigráfica': 1119, 'Climatológica Ordinaria': 912, 'Climatológica Principal': 804, 'Pluviográfica': 583, 'Meteorológica Especial': 226, 'Agrometeorológica': 130, 'Meteorologica Marina': 51, 'Sinóptica Principal': 35, 'Radio Sonda': 14, 'Sinóptica Secundaria': 14, 'Ambiental': 7, 'HidroMeteorologica': 6}

🔍 Estaciones fuera de Colombia: 13

🧹 Limpiando Estaciones IDEAM...
  Filas eliminadas: 13

✅ Estaciones limpias:
  Total: 9,672
  Activas: 5,641
  Nota: DIVIPOLA se asignará por cruce geoespacial en pipeline_integration

💾 

WindowsPath('../data/02_intermediate/normales_climatologicas_limpia.parquet')

Estaciones IDEAM:

13 estaciones fuera de Colombia eliminadas ✅

5.643 activas de 9.672 — buena cobertura

4 períodos de normales disponibles — usaremos el más reciente (1991-2020)

1 duplicado exacto eliminado ✅

Normales Climatológicas:

Outlier de 13.381 mm anuales de precipitación — es real, corresponde a zonas del Chocó que son de las más lluviosas del mundo. No se elimina.

6 parámetros disponibles — precipitación y temperatura son los más relevantes para el IRA

4 períodos históricos — para el IRA usaremos 1991-2020 como baseline

# NBI e IPM

In [9]:
print('='*60)
print('FUENTE 8 — NBI Censo 2018')
print('='*60)

url_nbi = 'https://www.dane.gov.co/files/censo2018/informacion-tecnica/CNPV-2018-NBI.xlsx'
r = requests.get(url_nbi, timeout=30)
df_nbi_raw = pd.read_excel(
    io.BytesIO(r.content), sheet_name="Municipios", header=None, skiprows=10
)
df_nbi_raw.columns = [
    "cod_depto","nombre_depto","cod_municipio","nombre_municipio",
    "nbi_total","miseria_total","vivienda_total","servicios_total",
    "hacinamiento_total","inasistencia_total","dependencia_total",
    "nbi_cabecera","miseria_cabecera","vivienda_cabecera","servicios_cabecera",
    "hacinamiento_cabecera","inasistencia_cabecera","dependencia_cabecera",
    "nbi_rural","miseria_rural","vivienda_rural","servicios_rural",
    "hacinamiento_rural","inasistencia_rural","dependencia_rural",
]

inspeccionar(df_nbi_raw, "NBI RAW")
analizar_outliers(df_nbi_raw, ["nbi_total","nbi_rural","miseria_total"], "NBI")

# Filas especiales
print(f'\n🔍 Filas especiales (totales/subtotales):')
print(df_nbi_raw[df_nbi_raw["cod_municipio"].isna() | (df_nbi_raw["cod_municipio"].astype(str).str.strip() == "")][["nombre_depto","nombre_municipio"]].head(5))

print(f'\n🧹 Limpiando NBI...')
df_nbi = df_nbi_raw.copy()
df_nbi = df_nbi.dropna(subset=["cod_municipio"])
df_nbi["cod_depto"] = df_nbi["cod_depto"].astype(float).astype(int).astype(str).str.zfill(2)
df_nbi["cod_municipio"] = df_nbi["cod_municipio"].astype(float).astype(int).astype(str).str.zfill(3)
df_nbi["divipola"] = df_nbi["cod_depto"] + df_nbi["cod_municipio"]
df_nbi = df_nbi[df_nbi["cod_municipio"] != "000"]
for col in ["nbi_total","nbi_cabecera","nbi_rural","miseria_total","miseria_rural"]:
    df_nbi[col] = pd.to_numeric(df_nbi[col], errors="coerce")
df_nbi = df_nbi.dropna(subset=["nbi_total"])

print(f'\n✅ NBI limpio:')
print(f'  Municipios: {len(df_nbi):,}')
print(f'  NBI total  — promedio: {df_nbi["nbi_total"].mean():.1f}% | max: {df_nbi["nbi_total"].max():.1f}%')
print(f'  NBI rural  — promedio: {df_nbi["nbi_rural"].mean():.1f}% | max: {df_nbi["nbi_rural"].max():.1f}%')
print(f'\n  Top 5 mayor NBI rural:')
print(df_nbi.nlargest(5,"nbi_rural")[["divipola","nombre_municipio","nbi_rural"]].to_string())

guardar_intermedio(df_nbi, "nbi_limpia")

print('\n' + '='*60)
print('FUENTE 9 — IPM Municipal 2018')
print('='*60)

ruta_ipm = RAW / 'anexo-censal-pobreza-municipal-2018.xlsx'
df_ipm_raw = pd.read_excel(ruta_ipm, sheet_name='4_IPM Mpio dominios', header=0, dtype={'ID': str})
df_ipm_raw.columns = ['divipola','municipio','ipm_total','ipm_cabecera','ipm_rural']

inspeccionar(df_ipm_raw, "IPM RAW")
analizar_outliers(df_ipm_raw, ["ipm_total","ipm_rural"], "IPM")

dup_ipm = df_ipm_raw.duplicated(subset=["divipola"]).sum()
print(f'\n🔍 Duplicados por DIVIPOLA: {dup_ipm:,}')

print(f'\n🧹 Limpiando IPM...')
df_ipm = df_ipm_raw.copy()
df_ipm = df_ipm.dropna(subset=['divipola'])
df_ipm['divipola'] = df_ipm['divipola'].str.zfill(5)
for col in ['ipm_total','ipm_cabecera','ipm_rural']:
    df_ipm[col] = pd.to_numeric(df_ipm[col], errors='coerce')
df_ipm = df_ipm.dropna(subset=['ipm_total'])

print(f'\n✅ IPM limpio:')
print(f'  Municipios: {len(df_ipm):,}')
print(f'  IPM total  — promedio: {df_ipm["ipm_total"].mean():.1f}% | max: {df_ipm["ipm_total"].max():.1f}%')
print(f'  IPM rural  — promedio: {df_ipm["ipm_rural"].mean():.1f}% | max: {df_ipm["ipm_rural"].max():.1f}%')
print(f'\n  Top 5 mayor IPM rural:')
print(df_ipm.nlargest(5,"ipm_rural")[["divipola","municipio","ipm_rural"]].to_string())

guardar_intermedio(df_ipm, "ipm_limpia")

FUENTE 8 — NBI Censo 2018

📋 NBI RAW
Shape: 1,125 filas × 25 columnas

Tipos de datos:
  cod_depto: object
  nombre_depto: object
  cod_municipio: float64
  nombre_municipio: object
  nbi_total: float64
  miseria_total: float64
  vivienda_total: float64
  servicios_total: float64
  hacinamiento_total: float64
  inasistencia_total: float64
  dependencia_total: float64
  nbi_cabecera: float64
  miseria_cabecera: float64
  vivienda_cabecera: float64
  servicios_cabecera: float64
  hacinamiento_cabecera: float64
  inasistencia_cabecera: float64
  dependencia_cabecera: float64
  nbi_rural: float64
  miseria_rural: float64
  vivienda_rural: float64
  servicios_rural: float64
  hacinamiento_rural: float64
  inasistencia_rural: float64
  dependencia_rural: float64

🔍 Nulos por columna:
                       nulos  pct_%
miseria_cabecera          22   1.96
vivienda_cabecera         22   1.96
dependencia_cabecera      22   1.96
inasistencia_cabecera     22   1.96
hacinamiento_cabecera     22   

WindowsPath('../data/02_intermediate/ipm_limpia.parquet')

NBI:

22 nulos en columnas de cabecera — son ANM que no tienen cabecera urbana. Correcto, no se eliminan.

2 filas especiales (totales nacionales) — eliminadas correctamente con el filtro cod_municipio != "000" ✅

1.122 municipios incluyendo ANM ✅

Los top 5 NBI rural son todos ANM de Vaupés y Guainía — confirma la decisión de incluirlos

IPM:

Sin nulos, sin duplicados — dataset limpio ✅

ipm_cabecera tiene tipo object en lugar de float — hay valores no numéricos. Hay que revisarlo.

Los mismos municipios ANM aparecen en el top IPM rural — coherente con NBI ✅

In [10]:
# Revisión ipm_cabecera — tipo object
print('🔍 Valores no numéricos en ipm_cabecera:')
ipm_cab_invalidos = df_ipm[pd.to_numeric(df_ipm['ipm_cabecera'], errors='coerce').isna()]
print(f'  Registros con valor no numérico: {len(ipm_cab_invalidos):,}')
if len(ipm_cab_invalidos) > 0:
    print(ipm_cab_invalidos[['divipola','municipio','ipm_cabecera']].head(10).to_string())

# Corregir
df_ipm['ipm_cabecera'] = pd.to_numeric(df_ipm['ipm_cabecera'], errors='coerce')
print(f'\n  Nulos después de corrección: {df_ipm["ipm_cabecera"].isna().sum():,}')
print(f'  (Esperado: ANM sin cabecera urbana)')

guardar_intermedio(df_ipm, "ipm_limpia")

🔍 Valores no numéricos en ipm_cabecera:
  Registros con valor no numérico: 20
     divipola         municipio  ipm_cabecera
1089    91263        EL ENCANTO           NaN
1090    91405       LA CHORRERA           NaN
1091    91407        LA PEDRERA           NaN
1092    91430       LA VICTORIA           NaN
1093    91460   MIRITÍ - PARANÁ           NaN
1094    91530    PUERTO ALEGRÍA           NaN
1095    91536      PUERTO ARICA           NaN
1097    91669  PUERTO SANTANDER           NaN
1098    91798          TARAPACÁ           NaN
1100    94343     BARRANCOMINAS           NaN

  Nulos después de corrección: 20
  (Esperado: ANM sin cabecera urbana)

💾 Guardado: ipm_limpia.parquet (0.0 MB) — 1,122 filas


WindowsPath('../data/02_intermediate/ipm_limpia.parquet')

Perfecto. Los 20 nulos en ipm_cabecera son exactamente los municipios de Amazonas y Guainía sin cabecera urbana formal — ANM y municipios rurales dispersos. Es un dato válido y esperable, no un error. Para el IRA usaremos ipm_rural que sí tiene cobertura completa.

# SIPSA-A, SIPSA-P y NOAA ONI

In [11]:
print('='*60)
print('FUENTE 10 — SIPSA-A 2026 I Cuatrimestre')
print('='*60)

ruta_sipsa_a = RAW / '2026 (I cuatrimestre).csv'
df_sipsa_a_raw = pd.read_csv(ruta_sipsa_a, sep=';', encoding='latin-1', dtype=str)

inspeccionar(df_sipsa_a_raw, "SIPSA-A RAW")

col_div = 'Divipola Municipio / ISO 3166-1 País Proc.'
print(f'\n🔍 Muestra DIVIPOLA raw: {df_sipsa_a_raw[col_div].head(5).tolist()}')
print(f'🔍 Grupos: {df_sipsa_a_raw["Grupo"].unique().tolist()}')
print(f'🔍 Fechas muestra: {df_sipsa_a_raw["FechaEncuesta"].head(3).tolist()}')

df_sipsa_a_raw["cant_num"] = pd.to_numeric(df_sipsa_a_raw["Cant Kg"], errors="coerce")
analizar_outliers(df_sipsa_a_raw, ["cant_num"], "SIPSA-A")

dup_sipsa = df_sipsa_a_raw.duplicated(subset=["FechaEncuesta", col_div, "Ali", "Fuente"]).sum()
print(f'\n🔍 Duplicados por fecha+municipio+alimento+fuente: {dup_sipsa:,}')

print(f'\n🧹 Limpiando SIPSA-A...')
df_sipsa_a = df_sipsa_a_raw.copy()
df_sipsa_a['divipola'] = df_sipsa_a[col_div].str.replace("'","").str.zfill(5)
df_sipsa_a['Cant Kg'] = pd.to_numeric(df_sipsa_a['Cant Kg'], errors='coerce')
df_sipsa_a['FechaEncuesta'] = pd.to_datetime(df_sipsa_a['FechaEncuesta'], errors='coerce', dayfirst=True)
df_sipsa_a['anio'] = df_sipsa_a['FechaEncuesta'].dt.year
df_sipsa_a['mes'] = df_sipsa_a['FechaEncuesta'].dt.month
antes = len(df_sipsa_a)
df_sipsa_a = df_sipsa_a.dropna(subset=['divipola','Cant Kg'])
df_sipsa_a = df_sipsa_a[df_sipsa_a['divipola'].str.len() == 5]
df_sipsa_a = df_sipsa_a[df_sipsa_a['Cant Kg'] > 0]
print(f'  Filas eliminadas: {antes - len(df_sipsa_a):,}')

print(f'\n✅ SIPSA-A limpio:')
print(f'  Registros: {len(df_sipsa_a):,}')
print(f'  Municipios origen: {df_sipsa_a["divipola"].nunique():,}')
print(f'  Grupos: {df_sipsa_a["Grupo"].unique().tolist()}')

guardar_intermedio(df_sipsa_a, "sipsa_a_limpia")

print('\n' + '='*60)
print('FUENTE 11 — SIPSA-P 2024')
print('='*60)

ruta_sipsa_p = RAW / 'mensual 24.csv'
df_sipsa_p_raw = pd.read_csv(ruta_sipsa_p, sep=';', encoding='utf-8-sig')

inspeccionar(df_sipsa_p_raw, "SIPSA-P RAW")
print(f'\n🔍 Mercados únicos: {df_sipsa_p_raw["Mercado"].nunique():,}')
print(f'🔍 Grupos: {df_sipsa_p_raw["Grupo"].unique().tolist()}')
print(f'🔍 Nota: Sin DIVIPOLA — variable temporal nacional en XGBoost')

df_sipsa_p_raw["precio_num"] = pd.to_numeric(
    df_sipsa_p_raw["Precio promedio por kilogramo*"].astype(str).str.replace('.','').str.replace(',','.'),
    errors='coerce'
)
analizar_outliers(df_sipsa_p_raw, ["precio_num"], "SIPSA-P")
dup_sipsa_p = df_sipsa_p_raw.duplicated(subset=["Fecha","Producto","Mercado"]).sum()
print(f'\n🔍 Duplicados por fecha+producto+mercado: {dup_sipsa_p:,}')

print(f'\n🧹 Limpiando SIPSA-P...')
df_sipsa_p = df_sipsa_p_raw.copy()
df_sipsa_p['precio_kg'] = df_sipsa_p["precio_num"]
antes = len(df_sipsa_p)
df_sipsa_p = df_sipsa_p.dropna(subset=['precio_kg'])
df_sipsa_p = df_sipsa_p[df_sipsa_p['precio_kg'] > 0]
print(f'  Filas eliminadas: {antes - len(df_sipsa_p):,}')

print(f'\n✅ SIPSA-P limpio:')
print(f'  Registros: {len(df_sipsa_p):,}')
print(f'  Mercados: {df_sipsa_p["Mercado"].nunique():,}')
print(f'  Precio promedio: ${df_sipsa_p["precio_kg"].mean():,.0f}/kg')

guardar_intermedio(df_sipsa_p, "sipsa_p_limpia")

print('\n' + '='*60)
print('FUENTE 12 — NOAA ONI')
print('='*60)

r = requests.get('https://www.cpc.ncep.noaa.gov/data/indices/oni.ascii.txt', timeout=15)
lineas = [l.split() for l in r.text.strip().split('\n')[1:] if l.strip()]
df_oni_raw = pd.DataFrame(lineas, columns=['season','year','total','anom'])

inspeccionar(df_oni_raw, "NOAA ONI RAW")

df_oni_raw["anom_num"] = pd.to_numeric(df_oni_raw["anom"], errors="coerce")
analizar_outliers(df_oni_raw, ["anom_num"], "ONI")
dup_oni = df_oni_raw.duplicated(subset=["year","season"]).sum()
print(f'\n🔍 Duplicados por año+temporada: {dup_oni:,}')

print(f'\n🧹 Limpiando ONI...')
df_oni = df_oni_raw.copy()
df_oni['anom'] = pd.to_numeric(df_oni['anom'], errors='coerce')
df_oni['total'] = pd.to_numeric(df_oni['total'], errors='coerce')
df_oni['year'] = pd.to_numeric(df_oni['year'], errors='coerce').astype('Int64')
df_oni['fase_enos'] = df_oni['anom'].apply(
    lambda x: 'El Niño' if x >= 0.5 else ('La Niña' if x <= -0.5 else 'Neutro')
)
df_oni = df_oni.dropna(subset=['anom'])

print(f'\n✅ ONI limpio:')
print(f'  Registros: {len(df_oni):,}')
print(f'  Años: {df_oni["year"].min()} - {df_oni["year"].max()}')
print(f'  Fases: {df_oni["fase_enos"].value_counts().to_dict()}')
print(df_oni.tail(4).to_string())

guardar_intermedio(df_oni, "oni_limpia")

FUENTE 10 — SIPSA-A 2026 I Cuatrimestre

📋 SIPSA-A RAW
Shape: 768,691 filas × 10 columnas

Tipos de datos:
  Fuente: object
  FechaEncuesta: object
  Divipola Depto Proc.: object
  Divipola Municipio / ISO 3166-1 País Proc.: object
  Departamento: object
  Municipio de Colombia / País Proc.: object
  Grupo: object
  Codigo CPC: object
  Ali: object
  Cant Kg: object

✅ Sin nulos

🔍 Duplicados exactos: 121,321

🔍 Muestra DIVIPOLA raw: ["'63001", "'52838", "'52838", "'52838", "'52838"]
🔍 Grupos: ['CARNES', 'TUBERCULOS, RAICES Y PLATANOS', 'FRUTAS', 'VERDURAS Y HORTALIZAS', 'GRANOS Y CEREALES', 'PROCESADOS', 'LACTEOS Y HUEVOS', 'PESCADOS']
🔍 Fechas muestra: ['2/01/2026', '2/01/2026', '2/01/2026']

🔍 Outliers (SIPSA-A):
  cant_num: 41,972 outliers | rango válido [-5300.00, 10220.00] | min: 1.00 | max: 40000.00

🔍 Duplicados por fecha+municipio+alimento+fuente: 398,682

🧹 Limpiando SIPSA-A...
  Filas eliminadas: 1,207

✅ SIPSA-A limpio:
  Registros: 767,484
  Municipios origen: 958
  Grupos

WindowsPath('../data/02_intermediate/oni_limpia.parquet')

SIPSA-A:

Apóstrofe en DIVIPOLA confirmado: '63001 → limpieza funcionó ✅

398.682 duplicados por llave — son registros del mismo alimento en el mismo día desde el mismo municipio hacia diferentes mercados. Se mantienen porque el flujo total es lo que importa

8 grupos alimentarios completos ✅

SIPSA-P:

Sin duplicados, sin nulos ✅

Precio máximo $191.528/kg — probablemente productos especiales o error. Se documenta pero no se elimina

Mismos 8 grupos que SIPSA-A — coherente ✅

NOAA ONI:

13 outliers son los eventos El Niño extremos históricos (1997-98, 2015-16) — datos reales, no se eliminan ✅

MAM 2026 ya en fase El Niño (0.51) — dato reciente muy relevante para el IRA

917 registros desde 1950 ✅

#  IPM Privaciones

In [12]:
print('='*60)
print('FUENTE 13 — IPM Privaciones 2018')
print('='*60)

df_priv_raw = pd.read_excel(
    ruta_ipm,
    sheet_name='6_Privaciones IPM Dpt-Mpio',
    header=1,
    dtype={'ID': str}
)

inspeccionar(df_priv_raw, "IPM Privaciones RAW")

dup_priv = df_priv_raw.duplicated(subset=['ID']).sum()
print(f'\n🔍 Duplicados por DIVIPOLA: {dup_priv:,}')
print(f'🔍 Privaciones disponibles: {list(df_priv_raw.columns[2:])}')

print(f'\n🧹 Limpiando IPM Privaciones...')
df_priv = df_priv_raw.copy()
df_priv = df_priv.dropna(subset=['ID'])
df_priv = df_priv.rename(columns={'ID': 'divipola', 'Municipio': 'municipio'})
df_priv['divipola'] = df_priv['divipola'].str.zfill(5)

for col in df_priv.columns[2:]:
    df_priv[col] = pd.to_numeric(df_priv[col], errors='coerce')

privaciones = list(df_priv.columns[2:])
analizar_outliers(df_priv, privaciones, "IPM Privaciones")

print(f'\n✅ IPM Privaciones limpio:')
print(f'  Municipios: {len(df_priv):,}')
print(f'  Privaciones: {len(privaciones)}')
print(f'\n  Muestra fila 1:')
print(df_priv.iloc[0].to_dict())

guardar_intermedio(df_priv, "ipm_privaciones_limpia")

FUENTE 13 — IPM Privaciones 2018

📋 IPM Privaciones RAW
Shape: 1,122 filas × 17 columnas

Tipos de datos:
  ID: object
  Municipio: object
  Analfabetismo: float64
  Bajo logro educativo: float64
  Barreras a servicios para cuidado de la primera infancia: float64
  Barreras de acceso a servicios de salud: float64
  Tasa de dependencia: float64
  Hacinamiento crítico: float64
  Inadecuada eliminación de excretas: float64
  Inasistencia escolar: float64
  Material inadecuado de paredes exteriores: float64
  Material inadecuado de pisos: float64
  Rezago escolar: float64
  Sin acceso a fuente de agua mejorada: float64
  Sin aseguramiento en salud: float64
  Trabajo infantil: float64
  Trabajo informal: float64

✅ Sin nulos

🔍 Duplicados exactos: 0

🔍 Duplicados por DIVIPOLA: 0
🔍 Privaciones disponibles: ['Analfabetismo', 'Bajo logro educativo', 'Barreras a servicios para cuidado de la primera infancia', 'Barreras de acceso a servicios de salud', 'Tasa de dependencia', 'Hacinamiento crític

WindowsPath('../data/02_intermediate/ipm_privaciones_limpia.parquet')

Sin nulos, sin duplicados ✅

1.122 municipios cobertura completa ✅

15 privaciones listas para XGBoost ✅

Los outliers son datos reales — municipios con condiciones extremas

Tres privaciones especialmente relevantes para el IRA:

Sin acceso a fuente de agua mejorada — max 99% (casi sin agua potable)

Inadecuada eliminación de excretas — max 98.7%

Trabajo informal — min 58.7%, todos los municipios tienen informalidad alta

# Aptitud Suelos muestra nacional

In [13]:
print('='*60)
print('FUENTE 14 — Aptitud Suelos UPRA (Nacionales)')
print('='*60)

DATASETS_NACIONALES = {
    "jdjx-qer4": ("cacao",         "Nacional"),
    "kwvf-nwea": ("cafe",          "Nacional"),
    "8fa5-z4v3": ("pina",          "Nacional"),
    "tx7u-frn2": ("aguacate_hass", "Nacional"),
    "ejwn-f7s3": ("pimenton",      "Nacional"),
    "yhkr-7mkb": ("aji_tabasco",   "Nacional"),
    "emsg-94di": ("fresa",         "Nacional"),
    "frjn-92um": ("maiz",          "Nacional_Tradicional"),
    "xt32-m7dh": ("mango",         "Nacional"),
    "ibc9-9f7c": ("arroz",         "Nacional_Secano"),
    "nxvg-ufyf": ("cebolla",       "Nacional_S2"),
    "urxm-qzje": ("papaya",        "Nacional"),
    "rcfj-3e57": ("banano",        "Nacional_Exportacion"),
    "p9xp-sm4v": ("cana_panelera", "Nacional"),
    "hixf-wnis": ("soya",          "Nacional_S2"),
    "jwn7-76wn": ("papa",          "Nacional_S1"),
    "s455-c4e6": ("papa",          "Nacional_S2"),
}

COLS_APT = ["cod_depart","cod_dane_m","aptitud","gridcode","area_ha"]
MAPEO_DIV = ["cod_dane_m","cod_dane_municipio","cod_mpio"]

dfs_apt = []
ok, error, sin_divipola = 0, 0, 0
client = Socrata("www.datos.gov.co", None, timeout=30)

for dataset_id, (cultivo, cobertura) in DATASETS_NACIONALES.items():
    nombre = f'{cultivo}_{cobertura}'
    try:
        total = int(client.get(dataset_id, select="count(*)")[0]["count"])
        registros = []
        offset = 0
        while offset < total:
            try:
                params = {"limit": 50000, "offset": offset, "order": ":id",
                         "select": ", ".join(COLS_APT)}
                registros.extend(client.get(dataset_id, **params))
            except:
                params = {"limit": 50000, "offset": offset, "order": ":id"}
                registros.extend(client.get(dataset_id, **params))
            offset += 50000
            time.sleep(0.2)

        df = pd.DataFrame.from_records(registros)
        col_div = next((c for c in MAPEO_DIV if c in df.columns), None)

        if col_div:
            df["divipola"] = df[col_div].astype(str).str.zfill(5)
            df["area_ha"] = pd.to_numeric(df.get("area_ha", pd.Series(dtype=float)), errors="coerce")
            df["gridcode"] = pd.to_numeric(df.get("gridcode", pd.Series(dtype=float)), errors="coerce")
            df["cultivo"] = cultivo
            df["cobertura"] = cobertura
            df = df.dropna(subset=["divipola","area_ha"])
            df = df[df["divipola"].str.len() == 5]

            muns = df["divipola"].nunique()
            cats = df["aptitud"].value_counts().to_dict() if "aptitud" in df.columns else {}
            nulos_apt = df["aptitud"].isna().sum()
            dup_apt = df.duplicated(subset=["divipola","aptitud"]).sum()

            print(f'✅ {nombre} ({dataset_id})')
            print(f'   Registros: {len(df):,} | Municipios: {muns:,}')
            print(f'   Categorías aptitud: {cats}')
            print(f'   Nulos en aptitud: {nulos_apt} | Duplicados divipola+aptitud: {dup_apt:,}')

            cols = ["divipola","aptitud","gridcode","area_ha","cultivo","cobertura"]
            dfs_apt.append(df[[c for c in cols if c in df.columns]])
            ok += 1
        else:
            print(f'⚠️  {nombre} ({dataset_id}) — sin DIVIPOLA directo')
            print(f'   Columnas disponibles: {list(df.columns)}')
            sin_divipola += 1

    except Exception as e:
        print(f'❌ {nombre} ({dataset_id}) — {e}')
        error += 1

client.close()

if dfs_apt:
    df_aptitud = pd.concat(dfs_apt, ignore_index=True)

    print(f'\n{"="*60}')
    print(f'Aptitud Nacional consolidada:')
    print(f'  Registros totales: {len(df_aptitud):,}')
    print(f'  Cultivos: {df_aptitud["cultivo"].nunique():,}')
    print(f'  Municipios únicos: {df_aptitud["divipola"].nunique():,}')
    print(f'\n  Distribución global de aptitud:')
    print(df_aptitud["aptitud"].value_counts().to_string())
    print(f'\n  Municipios por cultivo:')
    print(df_aptitud.groupby("cultivo")["divipola"].nunique().sort_values(ascending=False).to_string())
    print(f'\n  ✅ {ok} OK | ⚠️ {sin_divipola} sin DIVIPOLA | ❌ {error} errores')

    guardar_intermedio(df_aptitud, "aptitud_nacional_limpia")

FUENTE 14 — Aptitud Suelos UPRA (Nacionales)
✅ cacao_Nacional (jdjx-qer4)
   Registros: 169,266 | Municipios: 1,122
   Categorías aptitud: {'No apta': 80446, 'Aptitud media': 30266, 'Aptitud alta': 23494, 'Exclusión legal': 19112, 'Aptitud baja': 15948}
   Nulos en aptitud: 0 | Duplicados divipola+aptitud: 164,755
✅ cafe_Nacional (kwvf-nwea)
   Registros: 508,129 | Municipios: 1,122
   Categorías aptitud: {'Aptitud alta': 240975, 'No apta': 158769, 'Aptitud media': 44576, 'Exclusión legal': 42356, 'Aptitud baja': 21453}
   Nulos en aptitud: 0 | Duplicados divipola+aptitud: 503,852
✅ pina_Nacional (8fa5-z4v3)
   Registros: 146,067 | Municipios: 1,122
   Categorías aptitud: {'No apta': 69090, 'Aptitud media': 24431, 'Exclusión legal': 19109, 'Aptitud baja': 18127, 'Aptitud alta': 15310}
   Nulos en aptitud: 0 | Duplicados divipola+aptitud: 141,585
✅ aguacate_hass_Nacional (tx7u-frn2)
   Registros: 96,712 | Municipios: 1,122
   Categorías aptitud: {'No apta': 56658, 'Exclusión legal': 191

17 cultivos × 1.122 municipios — cobertura nacional completa ✅

0 nulos en aptitud — dato limpio de origen ✅

0 errores — todos los datasets nacionales accesibles ✅

5 categorías consistentes en todos los cultivos: Alta, Media, Baja, No apta, Exclusión legal ✅

Los duplicados altos por divipola+aptitud son normales — cada municipio tiene múltiples polígonos por categoría. En feature engineering los agruparemos sumando area_ha por municipio, cultivo y categoría de aptitud.

Hallazgo relevante: El 46.6% del área total es "No apta" y 12.7% es "Exclusión legal" — solo el 29% tiene aptitud media o alta. Eso es un dato crítico para el índice.

# Cruce geoespacial IDEAM

In [18]:
import sys
import importlib

sys.path.insert(0, '..')

# Forzar recarga del módulo
if 'src.pipeline_integration' in sys.modules:
    del sys.modules['src.pipeline_integration']
if 'src' in sys.modules:
    del sys.modules['src']

from src.pipeline_integration import cruzar_estaciones_municipios, cruzar_normales_municipios

print('✅ Módulo cargado correctamente')

✅ Módulo cargado correctamente


In [21]:
import sys
sys.path.insert(0, '..')
from src.pipeline_integration import cruzar_estaciones_municipios, cruzar_normales_municipios

print('='*60)
print('FUENTE 15 — Cruce Geoespacial Estaciones IDEAM')
print('='*60)

df_est_div = cruzar_estaciones_municipios()
print(f'\n✅ Estaciones con DIVIPOLA:')
print(f'  Total: {len(df_est_div):,}')
print(f'  Con DIVIPOLA: {df_est_div["divipola"].notna().sum():,}')
print(f'  Sin DIVIPOLA: {df_est_div["divipola"].isna().sum():,}')
print(f'  Municipios cubiertos: {df_est_div["divipola"].nunique():,}')

print('\n' + '='*60)
print('FUENTE 16 — Normales Climatológicas con DIVIPOLA')
print('='*60)

df_norm_div = cruzar_normales_municipios()
print(f'\n✅ Normales con DIVIPOLA:')
print(f'  Total 1991-2020: {len(df_norm_div):,}')
print(f'  Con DIVIPOLA: {df_norm_div["divipola"].notna().sum():,}')
print(f'  Municipios cubiertos: {df_norm_div["divipola"].nunique():,}')
print(f'  Parámetros: {df_norm_div["parametro"].unique().tolist()}')

print('📝 Nota metodológica:')
print('  Mapiripana (94663) es ANM de Guainía no incluida en MGN 2024.')
print('  Tiene datos NBI e IPM pero no participa en cruces geoespaciales.')
print('  Para el IRA se mantiene con indicadores socioeconómicos disponibles.')
print('  Cobertura geoespacial efectiva: 1.121/1.122 municipios (99.9%)')

FUENTE 15 — Cruce Geoespacial Estaciones IDEAM
2026-07-22 23:20:34 | INFO     | src.pipeline_integration:109 - Cruzando estaciones IDEAM con polígonos municipales...
2026-07-22 23:20:34 | INFO     | src.pipeline_integration:52 - Cargando polígonos municipales desde cache...
2026-07-22 23:20:34 | INFO     | src.pipeline_integration:123 - Estaciones: 9,672 | Municipios: 1,121
2026-07-22 23:20:35 | INFO     | src.pipeline_integration:130 - Con DIVIPOLA: 9,582 | Sin DIVIPOLA: 90
2026-07-22 23:20:35 | INFO     | src.pipeline_integration:135 - Guardado: estaciones_ideam_con_divipola.parquet

✅ Estaciones con DIVIPOLA:
  Total: 9,672
  Con DIVIPOLA: 9,582
  Sin DIVIPOLA: 90
  Municipios cubiertos: 1,033

FUENTE 16 — Normales Climatológicas con DIVIPOLA
2026-07-22 23:20:35 | INFO     | src.pipeline_integration:150 - Cruzando normales climatológicas con municipios...
2026-07-22 23:20:35 | INFO     | src.pipeline_integration:106 - Cargando estaciones con DIVIPOLA desde cache...
2026-07-22 23:20:

# Red Vial OSM (muestra)

In [22]:
from src.pipeline_integration import calcular_densidad_vial

print('='*60)
print('FUENTE 17 — Red Vial OSM (muestra 10 municipios)')
print('='*60)

# Muestra representativa — capital + municipios rurales
MUESTRA = [
    "05001",  # Medellín
    "25001",  # Agua de Dios
    "52001",  # Pasto
    "94001",  # Inírida
    "91001",  # Leticia
    "17001",  # Manizales
    "76001",  # Cali
    "15001",  # Tunja
    "68001",  # Bucaramanga
    "05591",  # Remedios (municipio rural Antioquia)
]

df_vial = calcular_densidad_vial(muestra_municipios=MUESTRA)

print(f'\n✅ Densidad vial calculada:')
print(f'  Municipios procesados: {len(df_vial):,}')
print(f'  Promedio densidad: {df_vial["densidad_vial_km_km2"].mean():.4f} km/km²')
print(f'  Max densidad: {df_vial["densidad_vial_km_km2"].max():.4f} km/km²')
print(f'  Min densidad: {df_vial["densidad_vial_km_km2"].min():.4f} km/km²')
print(f'\nDetalle:')
print(df_vial[["divipola","municipio","area_km2","longitud_vial_km","densidad_vial_km_km2"]].to_string())

FUENTE 17 — Red Vial OSM (muestra 10 municipios)
2026-07-22 23:20:44 | INFO     | src.pipeline_integration:197 - Calculando densidad vial rural por municipio (OSM)...
2026-07-22 23:20:44 | INFO     | src.pipeline_integration:52 - Cargando polígonos municipales desde cache...
2026-07-22 23:20:44 | INFO     | src.pipeline_integration:202 - Procesando muestra: 10 municipios
2026-07-22 23:20:55 | WARNING  | src.pipeline_integration:246 -   Sin red vial: PUERTO TRIUNFO (05591) — No data elements in server response. Check query location/filters and log.
2026-07-22 23:22:35 | WARNING  | src.pipeline_integration:246 -   Sin red vial: AGUA DE DIOS (25001) — No data elements in server response. Check query location/filters and log.
2026-07-22 23:34:40 | WARNING  | src.pipeline_integration:246 -   Sin red vial: INÍRIDA (94001) — HTTPSConnectionPool(host='overpass-api.de', port=443): Max retries exceeded with url: /api/interpreter (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection 

Funciona pero hay 3 municipios con área 0 — Puerto Triunfo, Agua de Dios e Inírida. El problema es que el cruce geoespacial a veces no encuentra el polígono correcto. Eso se resuelve en el pipeline completo con un fallback por nombre.
Los resultados que sí funcionaron tienen sentido:

Medellín 0.59 km/km² — ciudad densa ✅
Leticia 0.01 km/km² — Amazonas, muy poco vial ✅
Pasto 0.03 km/km² — municipio grande ✅

El módulo OSM funciona correctamente para el 70% de los casos en la muestra. El pipeline completo con los 1.121 municipios se correrá en background — puede tardar varias horas.

# Resumen de archivos intermedios

In [23]:
from pathlib import Path
import pandas as pd
import geopandas as gpd
import pyarrow.parquet as pq

INTERMEDIATE = Path('../data/02_intermediate')

print('='*60)
print('RESUMEN FINAL — data/02_intermediate')
print('='*60)

archivos = sorted(INTERMEDIATE.glob('*.parquet'))
total_mb = 0
resumen = []

for archivo in archivos:
    mb = archivo.stat().st_size / (1024*1024)
    total_mb += mb
    try:
        df_check = pd.read_parquet(archivo)
        filas = len(df_check)
        cols = len(df_check.columns)
    except:
        df_check = gpd.read_parquet(archivo)
        filas = len(df_check)
        cols = len(df_check.columns)

    tiene_divipola = 'divipola' in df_check.columns
    resumen.append({
        'Archivo': archivo.name,
        'Filas': f'{filas:,}',
        'Cols': cols,
        'MB': round(mb, 1),
        'DIVIPOLA': '✅' if tiene_divipola else '⚠️'
    })

df_res = pd.DataFrame(resumen)
print(df_res.to_string(index=False))
print(f'\nTotal: {len(archivos)} archivos | {total_mb:.1f} MB')
print(f'\n✅ Fase 3 — Preparación de Datos completada')
print(f'➡️  Siguiente: notebook 03_analisis_descriptivo.ipynb')
print(f'   Construcción tabla maestra municipio_features')

RESUMEN FINAL — data/02_intermediate
                               Archivo     Filas  Cols    MB DIVIPOLA
       aptitud_nacional_limpia.parquet 3,026,064     6 24.90        ✅
             densidad_vial_osm.parquet        10     5  0.00        ✅
        distritos_riego_limpia.parquet       580    13  0.00        ✅
 estaciones_ideam_con_divipola.parquet     9,672    15  0.60        ✅
       estaciones_ideam_limpia.parquet     9,672    12  0.60       ⚠️
                    eva_limpia.parquet    93,486    13  1.40        ✅
                finagro_limpia.parquet 1,906,481    12 29.90        ✅
      frontera_agricola_limpia.parquet   455,143     8  6.80        ✅
           informalidad_limpia.parquet        32     7  0.00       ⚠️
                    ipm_limpia.parquet     1,122     5  0.00        ✅
        ipm_privaciones_limpia.parquet     1,122    17  0.10        ✅
                    nbi_limpia.parquet     1,122    26  0.30        ✅
normales_climatologicas_limpia.parquet    14,693    2